#### model training

In [1]:
## import data and requirements package
#basic import 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
## modeling
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier
from xgboost import XGBClassifier
# from catboost import CatBoostClassifier

from sklearn.model_selection import train_test_split
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import recall_score,f1_score,precision_score,accuracy_score
from imblearn.over_sampling import SMOTE
import warnings

In [2]:
## loat the data 
df = pd.read_csv('data\Clean_data.csv')

In [3]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,TechSupport_Yes,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,1,29.85,29.85,0,...,0,0,0,0,0,0,0,0,1,0
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,0,0,0,0,0,1,0,0,0,1
2,1,0,0,0,2,1,1,53.85,108.15,1,...,0,0,0,0,0,0,0,0,0,1
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,1,0,0,0,0,1,0,0,0,0
4,0,0,0,0,2,1,1,70.70,151.65,1,...,0,0,0,0,0,0,0,0,1,0


In [4]:
## prepare X and y 
X = df.drop(columns=['Churn'])
y = df['Churn']

In [5]:
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True) * 100)

Churn
0    5174
1    1869
Name: count, dtype: int64
Churn
0    73.463013
1    26.536987
Name: proportion, dtype: float64


In [6]:
## split data into the train and test 
X_train,X_test,y_train,y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [7]:
X_train.shape,y_train.shape

((5634, 30), (5634,))

In [8]:
## handle imbalaced data
smote = SMOTE(random_state=42)
X_train_resampled , y_train_resampled = smote.fit_resample(X_train,y_train)

In [12]:
X_train_resampled.shape,y_train_resampled.shape

((8278, 30), (8278,))

In [9]:
models = {
    'LogisticRegression' : LogisticRegression(max_iter=1000),
    'Support Vector machine' : SVC(),
    'KNeighborsClassifier' : KNeighborsClassifier(), 
    'DecisionTreeClassifier' : DecisionTreeClassifier(),
    'RandomForestClassifier' : RandomForestClassifier(),
    'XGBClassifier' : XGBClassifier(),
    'AdaBoostClassifier' : AdaBoostClassifier()
}

In [10]:
def evaluate_model(true, predicted):
    accuracy = accuracy_score(true, predicted)
    precision = precision_score(true, predicted)
    recall = recall_score(true, predicted)
    f1 = f1_score(true, predicted)

    return accuracy, precision, recall, f1

In [14]:
model_list = []
f1_data = []
for name, model in models.items():

    model.fit(X_train_resampled, y_train_resampled)

    # Make predictions
    y_train_pred = model.predict(X_train_resampled)
    y_test_pred = model.predict(X_test)

    model_test_accuracy, model_test_precision, model_test_recall, model_test_f1 = evaluate_model(
        y_test,
        y_test_pred
    )
    model_train_accuracy, model_train_precision, model_train_recall, model_train_f1 = evaluate_model(
            y_train_resampled,
            y_train_pred
        )
    
    model_list.append(model)
    print(f"\n{name}")
    print('Model performance for Training set')
    print(f"Accuracy  : {model_train_accuracy:.4f}")
    print(f"Precision : {model_train_precision:.4f}")
    print(f"Recall    : {model_train_recall:.4f}")
    print(f"F1 Score  : {model_train_f1:.4f}")

    print('-'*50)

    print('Model performance for Test set')
    print(f"Accuracy  : {model_test_accuracy:.4f}")
    print(f"Precision : {model_test_precision:.4f}")
    print(f"Recall    : {model_test_recall:.4f}")
    print(f"F1 Score  : {model_test_f1:.4f}")

    print('='*50)
    f1_data.append(model_test_f1)




LogisticRegression
Model performance for Training set
Accuracy  : 0.8306
Precision : 0.8216
Recall    : 0.8446
F1 Score  : 0.8330
--------------------------------------------------
Model performance for Test set
Accuracy  : 0.7771
Precision : 0.5688
Recall    : 0.6631
F1 Score  : 0.6123

Support Vector machine
Model performance for Training set
Accuracy  : 0.6493
Precision : 0.6684
Recall    : 0.5927
F1 Score  : 0.6282
--------------------------------------------------
Model performance for Test set
Accuracy  : 0.6920
Precision : 0.4434
Recall    : 0.6283
F1 Score  : 0.5199

KNeighborsClassifier
Model performance for Training set
Accuracy  : 0.8426
Precision : 0.8065
Recall    : 0.9014
F1 Score  : 0.8513
--------------------------------------------------
Model performance for Test set
Accuracy  : 0.6955
Precision : 0.4492
Recall    : 0.6497
F1 Score  : 0.5311

DecisionTreeClassifier
Model performance for Training set
Accuracy  : 0.9984
Precision : 0.9998
Recall    : 0.9971
F1 Score  :

In [16]:
pd.DataFrame(list(zip(model_list, f1_data)), columns=['Model Name', 'f1_Score']).sort_values(by=["f1_Score"],ascending=False)

,Model Name,f1_Score
0,LogisticRegression(max_iter=1000),0.612346
6,"(DecisionTreeClassifier(max_depth=1, random_st...",0.601790
5,"XGBClassifier(base_score=None, booster=None, c...",0.597938
4,"(DecisionTreeClassifier(max_features='sqrt', r...",0.569571
2,KNeighborsClassifier(),0.531148
1,SVC(),0.519912
3,DecisionTreeClassifier(),0.512315
